In [1]:
import os
import scanpy as sc
import anndata as ad
import pandas as pd
import time 
import numpy as np
import liana as li
import itertools
import re # spliting the data 
import scipy as sci
from scipy.stats import false_discovery_control
import seaborn as sns
import matplotlib.pyplot as plt

# Analysis Start here 

In [2]:
res1 = pd.read_csv("/projects/bioinformatics/DB/scRNAseq_parkinson/liana_results_sample_modified.csv")

In [3]:
res1

,disease,ligand,ligand_complex,ligand_means,ligand_props,receptor,receptor_complex,receptor_means,receptor_props,source,target,lr_means,cellphone_pvals
0,normal,NLGN1,NLGN1,5.963885,0.999125,NRXN3,NRXN3,6.199426,0.999746,oligodendrocyte precursor cell,GABAergic neuron,6.081656,0.0
1,normal,NLGN1,NLGN1,5.963885,0.999125,NRXN1,NRXN1,5.933232,0.999083,oligodendrocyte precursor cell,oligodendrocyte precursor cell,5.948559,0.0
2,normal,LRRTM4,LRRTM4,5.563381,0.997374,NRXN3,NRXN3,6.199426,0.999746,oligodendrocyte precursor cell,GABAergic neuron,5.881403,0.0
3,normal,NLGN1,NLGN1,5.963885,0.999125,NRXN1,NRXN1,5.689255,0.999367,oligodendrocyte precursor cell,glutamatergic neuron,5.826570,0.0
4,normal,NRG3,NRG3,6.036231,0.998887,ERBB4,ERBB4,5.509386,0.921737,astrocyte,GABAergic neuron,5.772809,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
65251,Parkinson disease,C5,C5,0.165941,0.103910,C5AR1,C5AR1,0.177606,0.111052,glutamatergic neuron,central nervous system neuron,0.171773,1.0
65252,Parkinson disease,WNT8A,WNT8A,0.156221,0.103258,SFRP1,SFRP1,0.187223,0.102392,central nervous system neuron,central nervous system neuron,0.171722,0.0
65253,Parkinson disease,PYY,PYY,0.170627,0.112251,NPY5R,NPY5R,0.168829,0.108387,central nervous system neuron,central nervous system neuron,0.169728,0.0
65254,Parkinson disease,HLA-A,HLA-A,0.175738,0.106003,CD8A,CD8A,0.160177,0.102710,glutamatergic neuron,glutamatergic neuron,0.167958,1.0


In [4]:
# apply FDR correction
res1["cellphone_fdr"] = false_discovery_control(
    res1["cellphone_pvals"].values,
    method="bh"
    
)

In [5]:
res1

,disease,ligand,ligand_complex,ligand_means,ligand_props,receptor,receptor_complex,receptor_means,receptor_props,source,target,lr_means,cellphone_pvals,cellphone_fdr
0,normal,NLGN1,NLGN1,5.963885,0.999125,NRXN3,NRXN3,6.199426,0.999746,oligodendrocyte precursor cell,GABAergic neuron,6.081656,0.0,0.0
1,normal,NLGN1,NLGN1,5.963885,0.999125,NRXN1,NRXN1,5.933232,0.999083,oligodendrocyte precursor cell,oligodendrocyte precursor cell,5.948559,0.0,0.0
2,normal,LRRTM4,LRRTM4,5.563381,0.997374,NRXN3,NRXN3,6.199426,0.999746,oligodendrocyte precursor cell,GABAergic neuron,5.881403,0.0,0.0
3,normal,NLGN1,NLGN1,5.963885,0.999125,NRXN1,NRXN1,5.689255,0.999367,oligodendrocyte precursor cell,glutamatergic neuron,5.826570,0.0,0.0
4,normal,NRG3,NRG3,6.036231,0.998887,ERBB4,ERBB4,5.509386,0.921737,astrocyte,GABAergic neuron,5.772809,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65251,Parkinson disease,C5,C5,0.165941,0.103910,C5AR1,C5AR1,0.177606,0.111052,glutamatergic neuron,central nervous system neuron,0.171773,1.0,1.0
65252,Parkinson disease,WNT8A,WNT8A,0.156221,0.103258,SFRP1,SFRP1,0.187223,0.102392,central nervous system neuron,central nervous system neuron,0.171722,0.0,0.0
65253,Parkinson disease,PYY,PYY,0.170627,0.112251,NPY5R,NPY5R,0.168829,0.108387,central nervous system neuron,central nervous system neuron,0.169728,0.0,0.0
65254,Parkinson disease,HLA-A,HLA-A,0.175738,0.106003,CD8A,CD8A,0.160177,0.102710,glutamatergic neuron,glutamatergic neuron,0.167958,1.0,1.0


In [6]:
# filter properly (those that are significant!)

significant_lr = res1[
    (res1["cellphone_fdr"] < 0.05) &
    (res1["ligand_props"] > 0.1) &
    (res1["receptor_props"] > 0.1)
]

In [7]:
significant_lr

,disease,ligand,ligand_complex,ligand_means,ligand_props,receptor,receptor_complex,receptor_means,receptor_props,source,target,lr_means,cellphone_pvals,cellphone_fdr
0,normal,NLGN1,NLGN1,5.963885,0.999125,NRXN3,NRXN3,6.199426,0.999746,oligodendrocyte precursor cell,GABAergic neuron,6.081656,0.0,0.0
1,normal,NLGN1,NLGN1,5.963885,0.999125,NRXN1,NRXN1,5.933232,0.999083,oligodendrocyte precursor cell,oligodendrocyte precursor cell,5.948559,0.0,0.0
2,normal,LRRTM4,LRRTM4,5.563381,0.997374,NRXN3,NRXN3,6.199426,0.999746,oligodendrocyte precursor cell,GABAergic neuron,5.881403,0.0,0.0
3,normal,NLGN1,NLGN1,5.963885,0.999125,NRXN1,NRXN1,5.689255,0.999367,oligodendrocyte precursor cell,glutamatergic neuron,5.826570,0.0,0.0
4,normal,NRG3,NRG3,6.036231,0.998887,ERBB4,ERBB4,5.509386,0.921737,astrocyte,GABAergic neuron,5.772809,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65248,Parkinson disease,HLA-F,HLA-F,0.192334,0.100483,CD8A,CD8A,0.160177,0.102710,GABAergic neuron,glutamatergic neuron,0.176255,0.0,0.0
65250,Parkinson disease,WNT8A,WNT8A,0.156221,0.103258,FRZB,FRZB,0.189166,0.113783,central nervous system neuron,central nervous system neuron,0.172694,0.0,0.0
65252,Parkinson disease,WNT8A,WNT8A,0.156221,0.103258,SFRP1,SFRP1,0.187223,0.102392,central nervous system neuron,central nervous system neuron,0.171722,0.0,0.0
65253,Parkinson disease,PYY,PYY,0.170627,0.112251,NPY5R,NPY5R,0.168829,0.108387,central nervous system neuron,central nervous system neuron,0.169728,0.0,0.0


In [8]:
df = significant_lr

In [9]:
df.groupby("disease").size()

disease
Parkinson disease    21751
normal               21682
dtype: int64

In [10]:
df.groupby(["disease", "source", "target"]).size().sort_values(ascending=False)

disease            source                         target                       
normal             central nervous system neuron  central nervous system neuron    444
Parkinson disease  central nervous system neuron  central nervous system neuron    434
normal             central nervous system neuron  glutamatergic neuron             434
Parkinson disease  central nervous system neuron  glutamatergic neuron             410
normal             central nervous system neuron  GABAergic neuron                 398
                                                                                  ... 
Parkinson disease  leukocyte                      endothelial cell                  26
                                                  ependymal cell                    22
normal             leukocyte                      ependymal cell                    15
                                                  oligodendrocyte                   14
Parkinson disease  leukocyte                      

In [11]:
df.groupby(["disease", "ligand", "receptor"]).size().sort_values(ascending=False)

disease            ligand   receptor
Parkinson disease  WNT5B    FZD6        108
normal             WNT5B    FZD6         99
Parkinson disease  BMP6     ACVR2A       98
normal             BMP6     ACVR2A       93
Parkinson disease  BMP6     ACVR2B       90
                                       ... 
normal             ICAM3    ITGAL         1
                            ITGB2         1
Parkinson disease  CDH5     CDH5          1
                   COL10A1  ITGA2         1
                   CBR1     PTGER3        1
Length: 2538, dtype: int64

# Matching the same interaction

In [12]:
interaction_cols = [
    "source",
    "target",
    "ligand",
    "receptor"
]

df["interaction"] = (
    df[interaction_cols]
    .astype(str)
    .agg("|".join, axis=1)
)

/var/tmp/pbs.441868.pbs01/ipykernel_299438/3501416028.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


## To see how many interactions exist in each condition

In [13]:
df.groupby("disease")["interaction"].nunique()

disease
Parkinson disease    20435
normal               20397
Name: interaction, dtype: int64

## interactions present in both normal and PD

In [14]:
normal_interactions = set(
    df.loc[df["disease"] == "normal", "interaction"]
)

pd_interactions = set(
    df.loc[df["disease"] == "Parkinson disease", "interaction"]
)

common_interactions = normal_interactions & pd_interactions

print("Normal:", len(normal_interactions))
print("PD:", len(pd_interactions))
print("Common:", len(common_interactions))

Normal: 20397
PD: 20435
Common: 18229


## calculate PD vs normal change

In [15]:
comparison = (
    df[df["interaction"].isin(common_interactions)]
    .pivot_table(
        index=["source", "target", "ligand", "receptor"],
        columns="disease",
        values="lr_means",
        aggfunc="mean"
    )
    .reset_index()
)

comparison["delta_lr"] = (
    comparison["Parkinson disease"]
    - comparison["normal"]
)

eps = 1e-6

comparison["log2FC"] = np.log2(
    (comparison["Parkinson disease"] + eps) /
    (comparison["normal"] + eps)
)

comparison.sort_values("log2FC").head(20)

disease,source,target,ligand,receptor,Parkinson disease,normal,delta_lr,log2FC
10283,ependymal cell,oligodendrocyte,LPAR3,ADGRE5,0.435072,0.828959,-0.393887,-0.930046
8809,ependymal cell,GABAergic neuron,ANGPT1,TEK,0.567514,1.065021,-0.497507,-0.908153
7481,central nervous system neuron,oligodendrocyte precursor cell,SST,SSTR1,0.307641,0.568597,-0.260955,-0.886154
6384,central nervous system neuron,ependymal cell,WNT8A,FRZB,0.249228,0.456817,-0.207589,-0.874146
9247,ependymal cell,central nervous system neuron,ANGPT1,TEK,0.611273,1.114476,-0.503203,-0.866476
10176,ependymal cell,mural cell,LPAR3,ADGRE5,0.485478,0.881011,-0.395533,-0.859753
3043,astrocyte,central nervous system neuron,SEMA4A,NRP1,0.484960,0.872093,-0.387133,-0.846615
9763,ependymal cell,glutamatergic neuron,ANGPT1,TEK,0.615402,1.104593,-0.489191,-0.843913
11387,glutamatergic neuron,central nervous system neuron,VEGFB,NRP1,0.424925,0.759489,-0.334564,-0.837820
936,GABAergic neuron,central nervous system neuron,VEGFB,NRP1,0.501262,0.892081,-0.390819,-0.831608


## Which interactions are decreased in PD?

In [16]:
comparison.sort_values("log2FC").head(30)

disease,source,target,ligand,receptor,Parkinson disease,normal,delta_lr,log2FC
10283,ependymal cell,oligodendrocyte,LPAR3,ADGRE5,0.435072,0.828959,-0.393887,-0.930046
8809,ependymal cell,GABAergic neuron,ANGPT1,TEK,0.567514,1.065021,-0.497507,-0.908153
7481,central nervous system neuron,oligodendrocyte precursor cell,SST,SSTR1,0.307641,0.568597,-0.260955,-0.886154
6384,central nervous system neuron,ependymal cell,WNT8A,FRZB,0.249228,0.456817,-0.207589,-0.874146
9247,ependymal cell,central nervous system neuron,ANGPT1,TEK,0.611273,1.114476,-0.503203,-0.866476
10176,ependymal cell,mural cell,LPAR3,ADGRE5,0.485478,0.881011,-0.395533,-0.859753
3043,astrocyte,central nervous system neuron,SEMA4A,NRP1,0.484960,0.872093,-0.387133,-0.846615
9763,ependymal cell,glutamatergic neuron,ANGPT1,TEK,0.615402,1.104593,-0.489191,-0.843913
11387,glutamatergic neuron,central nervous system neuron,VEGFB,NRP1,0.424925,0.759489,-0.334564,-0.837820
936,GABAergic neuron,central nervous system neuron,VEGFB,NRP1,0.501262,0.892081,-0.390819,-0.831608


## Which interactions are increased?

In [17]:
comparison.sort_values("log2FC", ascending=False).head(30)

disease,source,target,ligand,receptor,Parkinson disease,normal,delta_lr,log2FC
9280,ependymal cell,central nervous system neuron,COL27A1,ITGA10,0.769439,0.418248,0.351191,0.879446
9492,ependymal cell,endothelial cell,COL27A1,ITGA10,1.137469,0.642068,0.495401,0.825029
9743,ependymal cell,ependymal cell,UBASH3B,ESR1,1.102945,0.640690,0.462255,0.783661
9798,ependymal cell,glutamatergic neuron,COL27A1,ITGA11,0.842766,0.499984,0.342782,0.753248
8842,ependymal cell,GABAergic neuron,COL27A1,ITGA1,0.857062,0.525627,0.331435,0.705359
15843,oligodendrocyte,ependymal cell,RELN,ITGA3,0.688487,0.423725,0.264762,0.700300
3116,astrocyte,endothelial cell,COL27A1,ITGA10,1.241891,0.782603,0.459288,0.666185
8264,endothelial cell,ependymal cell,TNXB,ITGA8,0.716391,0.457518,0.258874,0.646919
2883,astrocyte,central nervous system neuron,COL27A1,ITGA10,0.873861,0.558783,0.315078,0.645114
3386,astrocyte,ependymal cell,UBASH3B,ESR1,1.107334,0.710829,0.396505,0.639516


look at ligand_props and receptor_props too

Don't only look at lr_means.

In [18]:
comparison = (
    df[df["interaction"].isin(common_interactions)]
    .pivot_table(
        index=["source", "target", "ligand", "receptor"],
        columns="disease",
        values=[
            "lr_means",
            "ligand_means",
            "ligand_props",
            "receptor_means",
            "receptor_props"
        ],
        aggfunc="mean"
    )
)

In [19]:
comparison

ligand_means  \
disease                                                                       Parkinson disease   
source                         target                         ligand receptor                     
GABAergic neuron               GABAergic neuron               ACHE   CHRM1             0.212063   
                                                                     CHRM2             0.212063   
                                                                     CHRM3             0.212063   
                                                              ANGPT1 TEK               0.659557   
                                                              ANGPT2 TEK               0.526444   
...                                                                                         ...   
oligodendrocyte precursor cell oligodendrocyte precursor cell WNT5A  FZD9              0.340598   
                                                                     LRP5              0.340598   
                                                                     MCAM              0.340598   
                                                                     ROR2              0.340598   
                                                                     SFRP4             0.340598   

                                                                                         \
disease                                                                          normal   
source                         target                         ligand receptor             
GABAergic neuron               GABAergic neuron               ACHE   CHRM1     0.246222   
                                                                     CHRM2     0.246222   
                                                                     CHRM3     0.246222   
                                                              ANGPT1 TEK       0.623081   
                                                              ANGPT2 TEK       0.666031   
...                                                                                 ...   
oligodendrocyte precursor cell oligodendrocyte precursor cell WNT5A  FZD9      0.392573   
                                                                     LRP5      0.392574   
                                                                     MCAM      0.392574   
                                                                     ROR2      0.392574   
                                                                     SFRP4     0.392574   

                                                                                   ligand_props  \
disease                                                                       Parkinson disease   
source                         target                         ligand receptor                     
GABAergic neuron               GABAergic neuron               ACHE   CHRM1             0.113104   
                                                                     CHRM2             0.113104   
                                                                     CHRM3             0.113104   
                                                              ANGPT1 TEK               0.243034   
                                                              ANGPT2 TEK               0.256424   
...                                                                                         ...   
oligodendrocyte precursor cell oligodendrocyte precursor cell WNT5A  FZD9              0.138634   
                                                                     LRP5              0.138634   
                                                                     MCAM              0.138634   
                                                                     ROR2              0.138634   
                                                                     SFRP4             0.138634   

                                  

## Which cell populations change their communication in Parkinson's?

Calculate the mean interaction score by source and target:

In [20]:
celltype_summary = (
    df.groupby(
        ["disease", "source", "target"]
    )["lr_means"]
    .agg(
        mean="mean",
        median="median",
        count="count"
    )
    .reset_index()
)

celltype_summary.head()

,disease,source,target,mean,median,count
0,Parkinson disease,GABAergic neuron,GABAergic neuron,1.438220,0.876363,317
1,Parkinson disease,GABAergic neuron,astrocyte,1.412555,1.124030,280
2,Parkinson disease,GABAergic neuron,central nervous system macrophage,1.117717,0.843121,123
3,Parkinson disease,GABAergic neuron,central nervous system neuron,1.375114,0.936008,356
4,Parkinson disease,GABAergic neuron,endothelial cell,1.307053,1.177849,209


# Does Parkinson's have fewer or more inferred communication interactions?

In [21]:
interaction_counts = (
    df.groupby("disease")
      .apply(lambda x: x[["source", "target", "ligand", "receptor"]]
             .drop_duplicates()
             .shape[0])
)

interaction_counts

/var/tmp/pbs.441868.pbs01/ipykernel_299438/2455070064.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


disease
Parkinson disease    20435
normal               20397
dtype: int64